<div class="alert alert-block alert-success">

# App 5: 数据预处理与 SFT 数据集构建 (Data Preprocessing & SFT Dataset Construction)

**项目:** FIT5196 A1 (Extended Part)  
**模块:** App 5 - 基于特定商户的 RAG 与指令微调问答系统 (RAG & SFT Q&A System)  
**作者:** Zihan Yin  
**日期:** 2026.02.23

</div>

**概览 (Overview):**  
本 Notebook 是专为 **App 5 (指令微调大语言模型)** 设计的 **ETL (抽取、转换、加载)** 与数据合成管道。

本模块的核心目标是将 Task 1 产出的基础数据，转换为适用于大模型监督微调 (Supervised Fine-Tuning, SFT) 的高质量问答对格式。流程中应用了基于启发式的文本过滤、抗幻觉混合抽样 (Hybrid Sampling)，并调用了商用大语言模型 API（DeepSeek）进行反向指令生成。

<div class="alert alert-block alert-info">

## 预处理管道架构 (Preprocessing Pipeline Architecture)

为了构建具备抗噪能力和严格溯源特性的 SFT 数据集，本 Notebook 实施了以下 6 步管道：

1.  **基础路径配置与库导入:** 建立支持 UNC 路径的运行环境，定义输入与输出目录。
2.  **评论数据 `gmap_reviews` 清洗 (应用软阈值):** 加载原始评论数据。剔除无文本数据，并应用 `word_count >= 10` 的软阈值过滤无信息量的极短评论，确立高质量 SFT 候选池。
3.  **商铺元数据 `gmap_metadata` 清洗 (保留 Fallback 能力):** 加载全局商铺信息。执行源头去重 (Deduplication) 以保证索引唯一性。保留所有有效商铺信息，为终端应用的查询兜底 (Fallback) 机制提供数据支撑。
4.  **SFT 数据组装测试 (混合抽样):** 实施 "3条最长 + 7条随机" 的混合抽样策略。该策略旨在提供深度上下文的同时引入真实检索场景中的随机噪音，增强模型的鲁棒性。
5.  **批量调用商业 API (模块化测试):** 将核心数据提取与 Prompt 组装逻辑封装为独立函数。调用商业 API 生成 5 条测试数据，验证 JSON 结构和约束指令的有效性。
6.  **正式生产与数据集划分:** 批量请求 1210 次 API，生成正式的指令微调数据集。最终按照 1000:200:10 的比例将其划分为训练集 (Train)、验证集 (Validation) 和测试集 (Test) 并落盘。

</div>

## Step 1: 基础路径配置与库导入 (Path Configuration & Imports)
**目标:** 定义输入/输出路径并处理操作系统限制。
* **UNC 路径:** 使用 `\\?\` 前缀绕过 Windows 路径长度限制。
* **目录结构:** 创建 `App5` 专属的数据输出目录 `../../01_data/08_data_for_app5/`。

In [ ]:
import pandas as pd
import os
import random

# 基础路径配置
INPUT_REVIEWS_PATH = '../../01_data/02_data_wrangled/Task1_for_GroupALL.ftr'
INPUT_METADATA_PATH = '../../01_data/01_data_raw/meta-California.json'
DATA_OUTPUT_DIR = '../../01_data/08_data_for_app5/' 

# Windows UNC 路径辅助函数
def get_unc_path(path):
    abs_path = os.path.abspath(path)
    if os.name == 'nt' and not abs_path.startswith('\\\\?\\'):
        return '\\\\?\\' + abs_path.replace('/', '\\')
    return abs_path

os.makedirs(get_unc_path(DATA_OUTPUT_DIR), exist_ok=True)
print("环境与路径配置完成。")

环境与路径配置完成。


## Step 2: 读取与清洗评论数据 `gmap_reviews` (Loading & Cleaning Reviews)
**目标:** 过滤无意义数据，建立高质量的评论候选池。
* **保留字段:** 提取 `gmap_id`, `user_id`, `review_text`, `word_count`。
* **清洗逻辑:** 移除 `review_text` 为空的行。
* **软阈值过滤:** 计算评论单词数，仅保留长度 `>= 10` 个单词的评论，以确保上下文信息量。

In [ ]:
print("正在加载评论数据...")
gmap_reviews = pd.read_feather(get_unc_path(INPUT_REVIEWS_PATH))
initial_len = len(gmap_reviews)

# 1. 剔除完全没有文本的行
gmap_reviews = gmap_reviews.dropna(subset=['review_text'])

# 2. 动态计算单词数 (App 5 专属特征)
gmap_reviews['word_count'] = gmap_reviews['review_text'].apply(lambda x: len(str(x).split()))

# 3. 设定阈值：保留字数 >= 10 的评论 (大约一句完整的话)
gmap_reviews = gmap_reviews[gmap_reviews['word_count'] >= 10]

# 4. 只保留需要的列
gmap_reviews = gmap_reviews[['gmap_id', 'user_id', 'review_text', 'word_count']]

# 获取拥有高质量长评论的商铺 ID 集合 (用于后续抽样)
valid_gmap_ids_for_sft = set(gmap_reviews['gmap_id'].unique())

print(f"评论清洗完毕。原始行数: {initial_len} -> 有效长文本(>=10词)行数: {len(gmap_reviews)}")
print(f"拥有有效长评论的商铺数量 (SFT候选池大小): {len(valid_gmap_ids_for_sft)}")

正在加载评论数据...
评论清洗完毕。原始行数: 5767463 -> 有效长文本(>=10词)行数: 2125904
拥有有效长评论的商铺数量 (SFT候选池大小): 29088


In [24]:
gmap_reviews

,gmap_id,user_id,review_text,word_count
0,0x54d46d1125349d73:0x2ab5724cf1cbc511,112420076785194463877,We met with friends at this restaurant for din...,54
1,0x54d46d1125349d73:0x2ab5724cf1cbc511,103079253096551262298,Best place on earth !!! Good staff cool little...,14
3,0x54d46d1125349d73:0x2ab5724cf1cbc511,117974159860031660818,Fun friendly good service new look a family of...,10
4,0x54d46d1125349d73:0x2ab5724cf1cbc511,100368141243486348290,Couldn't find a better bartender than Connie s...,16
5,0x54d46d1125349d73:0x2ab5724cf1cbc511,102355600676196912732,This was a great rest stop after what seemed l...,48
...,...,...,...,...
7671327,0x80ecf0c931c4942d:0x3825f344a8f7bb1,112091978081500321537,Travis was the guy who picked me up and man du...,37
7671328,0x80ecf0c931c4942d:0x3825f344a8f7bb1,113058476269158003964,amazing service! gave ride to chase after find...,15
7671332,0x80ecf0c931c4942d:0x3825f344a8f7bb1,113752354153628494905,They seriously suck. Had me stuck in SLO for ...,24
7671337,0x80ecf0c931c4942d:0x3825f344a8f7bb1,102123433907360189909,These people are the lowestWorst customer serv...,17


## Step 3: 读取全局商铺元数据 `gmap_metadata` (Loading Metadata & Deduplication)
**目标:** 加载商铺特征，并处理数据冗余。
* **保留字段:** 提取 `gmap_id`, `name`, `category`, `description`, `address`, `avg_rating`, `url`。
* **源头去重:** 针对 `gmap_id` 执行去重 (`drop_duplicates`)，确保后续索引查找的唯一性和代码稳定性。
* **Fallback 机制:** 保留全量有效商铺，不与评论数据执行强制内连接 (Inner Join)，为系统提供基础信息兜底能力。

In [25]:
print("正在加载商铺元数据...")

# 1. 读取全部 JSON Lines
gmap_metadata = pd.read_json(get_unc_path(INPUT_METADATA_PATH), lines=True)
initial_meta_len = len(gmap_metadata)

# 2. 保留指定的 7 个列 (包含用于后续 Fallback 兜底的 address, avg_rating, url)
cols_to_keep = ['gmap_id', 'name', 'category', 'description', 'address', 'avg_rating', 'url']
gmap_metadata = gmap_metadata[cols_to_keep]

# 3. 仅清除 name 为空的绝对脏数据
gmap_metadata = gmap_metadata.dropna(subset=['name'])

# 4. 【核心修复】在源头移除重复的 gmap_id，保留最后一条有效记录
# 这样可以保证后续将其设为索引时，索引是 100% 唯一的
gmap_metadata = gmap_metadata.drop_duplicates(subset=['gmap_id'], keep='last')

# 5. 将 gmap_id 设为索引，极大提升后续按 ID 查找信息的速度
gmap_metadata = gmap_metadata.set_index('gmap_id')

print(f"元数据加载完毕。原始商铺数: {initial_meta_len} -> 有效全局商铺数(已去重): {len(gmap_metadata)}")

正在加载商铺元数据...
元数据加载完毕。原始商铺数: 515961 -> 有效全局商铺数(已去重): 513124


In [26]:
gmap_metadata

,name,category,description,address,avg_rating,url
gmap_id,,,,,,
0x80c2c98c0e3c16fd:0x29ec8a728764fdf9,City Textile,[Textile exporter],None,"City Textile, 3001 E Pico Blvd, Los Angeles, C...",4.5,https://www.google.com/maps/place//data=!4m2!3...
0x80c2c778e3b73d33:0xbdc58662a4a97d49,San Soo Dang,[Korean restaurant],None,"San Soo Dang, 761 S Vermont Ave, Los Angeles, ...",4.4,https://www.google.com/maps/place//data=!4m2!3...
0x80c2c89923b27a41:0x32041559418d447,Nova Fabrics,[Fabric store],None,"Nova Fabrics, 2200 E 11th St, Los Angeles, CA ...",3.3,https://www.google.com/maps/place//data=!4m2!3...
0x80c2c632f933b073:0xc31785961fe826a6,Nobel Textile Co,[Fabric store],None,"Nobel Textile Co, 719 E 9th St, Los Angeles, C...",4.3,https://www.google.com/maps/place//data=!4m2!3...
0x80c2cf163db6bc89:0x219484e2edbcfa41,Matrix International Textiles,[Fabric store],None,"Matrix International Textiles, 1363 S Bonnie B...",3.5,https://www.google.com/maps/place//data=!4m2!3...
...,...,...,...,...,...,...
0x80dd4a7afea27289:0xe49cfab49567e5cb,McDonald's,"[Fast food restaurant, Breakfast restaurant, C...","Classic, long-running fast-food chain known fo...","McDonald's, 1728 Lomita Blvd, Lomita, CA 90717",4.1,https://www.google.com/maps/place//data=!4m2!3...
0x80dcba7983a059af:0x2a006c069483d3d2,California Citrus State Historic Park,"[Park, Tourist attraction]",Park dedicated to preserving the history of Ca...,"California Citrus State Historic Park, 9400 Du...",4.7,https://www.google.com/maps/place//data=!4m2!3...
0x80dcb09e3af6228b:0xa55fc2f742364e02,California Citrus,[State park],None,"California Citrus, 1999 Van Buren Boulevard, R...",4.8,https://www.google.com/maps/place//data=!4m2!3...


## Step 4: SFT 数据组装测试 (Hybrid Sampling Test)
**目标:** 验证数据拼接逻辑与混合抽样策略的有效性。
* **混合抽样 (Hybrid Sampling):** 对单一商铺的评论按长度排序，抽取前 3 条最长评论（保证信息深度），并随机抽取 7 条其余评论（模拟 RAG 检索噪音），最后打乱顺序。
* **动态 Prompt:** 根据缺失值情况（如 `category` 或 `description` 为空），动态调整传入大模型的上下文内容。

In [ ]:
print("="*50)
print("抽取一家店铺进行混合抽样 (3条最长 + 7条随机) 验证")
print("="*50)

def get_safe_val(val):
    """安全读取 DataFrame 字段的辅助函数，解决空值或列表导致的真值歧义报错。"""
    if val is None: return None
    if isinstance(val, (float, int)) and pd.isna(val): return None
    if isinstance(val, list): return ", ".join([str(x) for x in val]) if val else None
    return str(val)

# 1. 从“SFT候选池”中筛选出至少有 10 条长评论的店铺
review_counts = gmap_reviews.groupby('gmap_id').size()
eligible_stores = review_counts[review_counts >= 10].index.tolist()

# 2. 随机抽取一家店
sample_gmap_id = random.choice(eligible_stores)
store_info = gmap_metadata.loc[sample_gmap_id]

# 3. 混合抽样逻辑 (Hybrid Sampling: 3 Top + 7 Random)
store_reviews_df = gmap_reviews[gmap_reviews['gmap_id'] == sample_gmap_id].sort_values(by='word_count', ascending=False)

if len(store_reviews_df) <= 10:
    hybrid_reviews = store_reviews_df['review_text'].tolist()
else:
    # 策略: 抽取 3 条最长的 (确保深度) + 7 条随机的 (模拟噪音，增加容错率)
    top_3_reviews = store_reviews_df.head(3)
    remaining_reviews = store_reviews_df.iloc[3:]
    random_7_reviews = remaining_reviews.sample(min(7, len(remaining_reviews)), random_state=42)
    
    # 合并并打乱顺序 (Shuffle)，防止大模型发现“长文本总在前面”的规律
    hybrid_df = pd.concat([top_3_reviews, random_7_reviews]).sample(frac=1, random_state=42)
    hybrid_reviews = hybrid_df['review_text'].tolist()

# 4. 安全提取需要打印的字段
store_name = get_safe_val(store_info['name'])
cat_str = get_safe_val(store_info['category'])
desc_str = get_safe_val(store_info['description'])

# 5. 模拟打印发给大模型的 User Prompt
print(f"Business Name: {store_name}")
if cat_str:
    print(f"Category: {cat_str}")
if desc_str:
    print(f"Description: {desc_str}")
    
print(f"\nReal Customer Reviews (Hybrid Sampled - Total: {len(hybrid_reviews)}):")
for i, rev in enumerate(hybrid_reviews, 1):
    # 控制台打印太长，做一下截断演示
    print(f"[{i}] {rev[:120]}..." if len(rev) > 120 else f"[{i}] {rev}")

抽取一家店铺进行混合抽样 (3条最长 + 7条随机) 验证
Business Name: Rockin' Baja Lobster - Old Town
Category: Mexican restaurant, Latin American restaurant, Meal delivery, Seafood market, Seafood restaurant
Description: Inspired by the dishes of Baja's Puerto Nuevo fishing town, this fish & lobster taco shop bustles.

Real Customer Reviews (Hybrid Sampled - Total: 10):
[1] Food is good but service sucks.  Have been a couple times due to friends in town.  First time, good food but service sup...
[2] Salty food, red pork enchiladas were horrible, Margaritas served in a cheap plastic cup, tasted like they use cheap wine...
[3] This place was great! Had great service, food and the atmosphere was cool. We were in staying in Old Town for the week a...
[4] I don’t like giving one star reviews, and I also don’t like reading one star reviews because most people who give a one ...
[5] Food was delicious and the people were very nice and we Had great service..
[6] Unfortunately, our dinner experience was no where near 

## Step 5: 批量调用 Deepseek API (Modularized API Testing)
**目标:** 封装核心逻辑，并进行小规模的 Deepseek API 联调测试。
* **函数封装:** 将数据提取、混合抽样、安全赋值和 API 交互逻辑封装至 `generate_single_sft_sample` 函数，提高代码复用率。
* **严格约束:** 实施带有负向约束 (Negative Constraint) 的 System Prompt，强制要求模型使用 `[1][2]` 标注引用，并禁止输出多余的样本量声明。
* **输出验证:** 生产 5 条测试数据保存为 `App5_Data_Test_Mini.jsonl`，用于人工检验问答质量与格式合规性。

统一发送给 Deepseek API 的 Prompt 如下所示： 

```{python}
SYSTEM_PROMPT = """
You are an expert data curator responsible for generating Instruction Fine-Tuning (SFT) data for a local business RAG system.
I will provide you with a business's basic information and a set of numbered, real customer reviews.

Your task is to:
1. Act as a curious customer and generate a realistic question based on the provided business profile and reviews.
2. Act as an objective, strictly factual AI assistant and answer the generated question SOLELY based on the provided reviews.

Strict Constraints (MUST FOLLOW):
1. **Strict Traceability**: Every claim in your answer MUST be directly supported by the provided reviews. Do NOT assume any services, items, or sentiments exist if they are not explicitly mentioned.
2. **Citation Requirement**: You must append citations in brackets at the end of relevant sentences (e.g., "Customers highly recommend the fried chicken [1][3].").
3. **Missing Metadata**: The 'Category' or 'Description' fields may be missing. If so, rely entirely on the business name and the reviews to infer context.
4. **Sample Size Awareness**: If I provide fewer than 5 reviews, your answer must acknowledge the limited data. However, if there are 5 or more reviews, you MUST NOT mention the sample size, review count, or add any concluding remarks about the data size.
5. **Unanswerable Queries (20% Probability)**: 20% of the time, you MUST generate a plausible question that is entirely UNANSWERABLE based on the provided reviews. When you do this, your answer MUST strictly be: "I'm sorry, but the provided customer reviews do not contain information regarding this."
6. **JSON Output Only**: Output strictly in valid JSON format with keys "generated_question" and "generated_answer". No markdown formatting.
"""
```

In [ ]:
import json
import traceback
import os
import random
from tqdm import tqdm
from openai import OpenAI

# ------------------------------------------
# 1. API 客户端与全局配置
# ------------------------------------------
DEEPSEEK_API_KEY = "sk-144d06ae3f5b491d8b866b41941021e1" 
client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com")
MODEL_NAME = "deepseek-chat" 

NUM_TEST_SAMPLES = 5
OUTPUT_TEST_PATH = os.path.join(get_unc_path(DATA_OUTPUT_DIR), 'App5_Data_Test_Mini.jsonl')

# 全英文 Prompt，强制大模型客观总结并标注引用
SYSTEM_PROMPT = """
You are an expert data curator responsible for generating Instruction Fine-Tuning (SFT) data for a local business RAG system.
I will provide you with a business's basic information and a set of numbered, real customer reviews.

Your task is to:
1. Act as a curious customer and generate a realistic question based on the provided business profile and reviews.
2. Act as an objective, strictly factual AI assistant and answer the generated question SOLELY based on the provided reviews.

Strict Constraints (MUST FOLLOW):
1. **Strict Traceability**: Every claim in your answer MUST be directly supported by the provided reviews. Do NOT assume any services, items, or sentiments exist if they are not explicitly mentioned.
2. **Citation Requirement**: You must append citations in brackets at the end of relevant sentences (e.g., "Customers highly recommend the fried chicken [1][3].").
3. **Missing Metadata**: The 'Category' or 'Description' fields may be missing. If so, rely entirely on the business name and the reviews to infer context.
4. **Sample Size Awareness**: If I provide fewer than 5 reviews, your answer must acknowledge the limited data. However, if there are 5 or more reviews, you MUST NOT mention the sample size, review count, or add any concluding remarks about the data size.
5. **Unanswerable Queries (20% Probability)**: 20% of the time, you MUST generate a plausible question that is entirely UNANSWERABLE based on the provided reviews. When you do this, your answer MUST strictly be: "I'm sorry, but the provided customer reviews do not contain information regarding this."
6. **JSON Output Only**: Output strictly in valid JSON format with keys "generated_question" and "generated_answer". No markdown formatting.
"""

# ------------------------------------------
# 2. 核心函数定义 (供当前测试及未来 Step 6 复用)
# ------------------------------------------
def generate_single_sft_sample(gmap_id, metadata_df, reviews_df, api_client, model_name, sys_prompt):
    """
    核心数据生成引擎：处理单家商铺的数据提取、混合抽样、API 调用及 JSON 组装。
    返回: 成功则返回 sft_sample 字典，失败则返回 None
    """
    try:
        # 1. 提取元数据 (得益于 Step 3 的源头去重，这里稳定返回单行 Series)
        store_info = metadata_df.loc[gmap_id]
        
        # 2. 混合抽样 (3最长 + 7随机)
        store_reviews_df = reviews_df[reviews_df['gmap_id'] == gmap_id].sort_values(by='word_count', ascending=False)
        if len(store_reviews_df) <= 10:
            hybrid_reviews = store_reviews_df['review_text'].tolist()
        else:

            top_3 = store_reviews_df.head(3)
            random_7 = store_reviews_df.iloc[3:].sample(min(7, len(store_reviews_df)-3))
            hybrid_reviews = pd.concat([top_3, random_7]).sample(frac=1)['review_text'].tolist()
            
        # 3. 安全提取所需字段
        store_name = get_safe_val(store_info['name'])
        cat_str = get_safe_val(store_info['category'])
        desc_str = get_safe_val(store_info['description'])
        
        # 4. 动态组装喂给 API 的 User Prompt
        user_prompt = f"Business Name: {store_name}\n"
        if cat_str: user_prompt += f"Category: {cat_str}\n"
        if desc_str: user_prompt += f"Description: {desc_str}\n"
        user_prompt += f"\nReal Customer Reviews (Total: {len(hybrid_reviews)}):\n"
        for i, rev in enumerate(hybrid_reviews, 1):
            user_prompt += f"[{i}] {rev}\n"
            
        # 5. 调用商业 API
        response = api_client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": sys_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_format={ "type": "json_object" }, # 强制 API 返回 JSON 字符串
            temperature=0.7 
        )
        
        # 6. 解析大模型吐出的 JSON
        api_result = json.loads(response.choices[0].message.content)
        
        # 7. 组装终极 SFT 格式 (外层 Metadata + 内层 messages)
        sft_sample = {
            "gmap_id": gmap_id,
            "store_name": store_name,
            "category": cat_str,
            "description": desc_str,
            "source_reviews_count": len(hybrid_reviews),
            "address": get_safe_val(store_info['address']),
            "avg_rating": get_safe_val(store_info['avg_rating']),
            "url": get_safe_val(store_info['url']),
            "messages": [
                {"role": "system", "content": "You are a professional local business analyst. Please objectively and accurately answer user questions about a specific business based on the provided context. Be honest if you don't know."},
                {"role": "user", "content": user_prompt + f"\nUser Question: {api_result['generated_question']}"},
                {"role": "assistant", "content": api_result['generated_answer']}
            ]
        }
        return sft_sample
        
    except Exception as e:
        # 在测试阶段打印出详细错误栈，方便定位问题；生产阶段可改为 pass
        print(f"\n[Error] Failed on gmap_id {gmap_id}: {str(e)}")
        traceback.print_exc()
        return None

In [ ]:
# ------------------------------------------
# 3. 执行测试主循环
# ------------------------------------------
print(f"开始生成 {NUM_TEST_SAMPLES} 条测试数据...")

# 提取测试用的候选商铺池 (多准备一些以防失败)
sample_stores = random.sample(list(eligible_stores), min(NUM_TEST_SAMPLES * 2, len(eligible_stores)))
success_count = 0

with open(OUTPUT_TEST_PATH, 'w', encoding='utf-8') as f:
    for gmap_id in tqdm(sample_stores, desc="Testing API"):
        if success_count >= NUM_TEST_SAMPLES:
            break
            
        # 高度解耦：直接调用上面定义好的核心函数
        sft_sample = generate_single_sft_sample(
            gmap_id=gmap_id, 
            metadata_df=gmap_metadata, 
            reviews_df=gmap_reviews, 
            api_client=client, 
            model_name=MODEL_NAME, 
            sys_prompt=SYSTEM_PROMPT
        )
        
        # 如果成功获取数据，则写入文件
        if sft_sample is not None:
            f.write(json.dumps(sft_sample, ensure_ascii=False) + '\n')
            success_count += 1

print(f"\n✅ 测试生成完毕！成功生成 {success_count} 条数据。请前往 {OUTPUT_TEST_PATH} 检查问答质量。")

开始生成 5 条测试数据...


Testing API:  50%|█████     | 5/10 [00:27<00:27,  5.50s/it]


✅ 测试生成完毕！成功生成 5 条数据。请前往 \\?\e:\05_YZH_DS\02_Monash_DS\2024_S2_FIT5196_Data wrangling\5. assessments\FIT5196 assignment 1\04_github_submission\2024-07-Google-Map-Review-Analysis-and-Machine-Learning\01_data\05_data_for_app5\App5_Data_Test_Mini.jsonl 检查问答质量。


In [39]:
# ==========================================
# 附加: 数据集审查与维度验证 (Dataset Inspection)
# ==========================================
import json
import os

def inspect_jsonl_file(file_path, num_lines_to_print=1):
    """
    读取 JSONL 文件，打印总行数，并美化输出指定数量的数据行。
    """
    if not os.path.exists(file_path):
        print(f"[Error] File not found: {file_path}")
        return

    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        total_rows = len(lines)
        
        print(f"File: {os.path.basename(file_path)}")
        print(f"Total rows (Dimension): {total_rows}")
        print("-" * 50)
        
        if total_rows > 0:
            for i in range(min(num_lines_to_print, total_rows)):
                sample_data = json.loads(lines[i])
                print(f"--- Sample {i+1} ---")
                print(json.dumps(sample_data, indent=4, ensure_ascii=False))
        print("=" * 60 + "\n")

In [42]:
# 确保路径与你生成测试文件的路径一致
test_file_path = os.path.join(get_unc_path(DATA_OUTPUT_DIR), 'App5_Data_Test_Mini.jsonl')

# 执行检查并打印第一条数据
inspect_jsonl_file(test_file_path, num_lines_to_print=1)

File: App5_Data_Test_Mini.jsonl
Total rows (Dimension): 5
--------------------------------------------------
--- Sample 1 ---
{
    "gmap_id": "0x80dcdf57959727fb:0x7293c9151e3e790f",
    "store_name": "GoMobile Tires Newport Beach",
    "category": "Tire shop",
    "description": null,
    "source_reviews_count": 10,
    "address": "GoMobile Tires Newport Beach, 3829 Birch St, Newport Beach, CA 92660",
    "avg_rating": "5.0",
    "url": "https://www.google.com/maps/place//data=!4m2!3m1!1s0x80dcdf57959727fb:0x7293c9151e3e790f?authuser=-1&hl=en&gl=us",
    "messages": [
        {
            "role": "system",
            "content": "You are a professional local business analyst. Please objectively and accurately answer user questions about a specific business based on the provided context. Be honest if you don't know."
        },
        {
            "role": "user",
            "content": "Business Name: GoMobile Tires Newport Beach\nCategory: Tire shop\n\nReal Customer Reviews (Total

## Step 6: 批量调用 Deepseek API (Production Generation & Split)
**目标:** 批量生产正式 SFT 数据集，并执行标准的数据集划分。
* **全量生产:** 循环调用 API 接口，收集 1210 条合法的 SFT 训练样本。
* **数据划分:** 将数据集整体打乱，按照 **Train (1000)**, **Validation (200)**, **Test (10)** 的规模进行切分。
* **数据落盘:** 序列化并导出为三个独立的 `.jsonl` 文件，供后续模型微调使用。

In [ ]:
import json
import random
from tqdm import tqdm
from openai import OpenAI

# ------------------------------------------
# 1. 切分目标配置
# ------------------------------------------
NUM_TRAIN = 1000
NUM_VAL = 200
NUM_TEST = 10
TOTAL_NEEDED = NUM_TRAIN + NUM_VAL + NUM_TEST # 1210

print(f"目标生成 SFT 数据总量: {TOTAL_NEEDED} 条...")

# 提取比需求多 1.5 倍的候选店铺，以防部分店铺在 API 调用时失败
sample_stores = random.sample(list(eligible_stores), min(int(TOTAL_NEEDED * 1.5), len(eligible_stores)))
all_sft_data = [] # 内存列表，暂存所有成功生成的 SFT 样本

# ------------------------------------------
# 2. 正式生产大循环 (复用 Step 5 定义的核心函数)
# ------------------------------------------
for gmap_id in tqdm(sample_stores, desc="Production Generation"):
    if len(all_sft_data) >= TOTAL_NEEDED:
        break # 如果收集到了 1210 条，提前终止循环
        
    # 调用高度解耦的核心引擎
    sft_sample = generate_single_sft_sample(
        gmap_id=gmap_id, 
        metadata_df=gmap_metadata, 
        reviews_df=gmap_reviews, 
        api_client=client, 
        model_name=MODEL_NAME, 
        sys_prompt=SYSTEM_PROMPT
    )
    
    if sft_sample is not None:
        all_sft_data.append(sft_sample)

# ------------------------------------------
# 4. 数据集随机打乱与拆分落盘
# ------------------------------------------
print(f"\nAPI 调用结束。成功收集 {len(all_sft_data)} 条数据。正在进行切分...")

# 再次全局打乱数据集，确保 Train/Val/Test 数据分布均匀
random.shuffle(all_sft_data)

# 切片拆分
train_data = all_sft_data[:NUM_TRAIN]
val_data = all_sft_data[NUM_TRAIN : NUM_TRAIN + NUM_VAL]
test_data = all_sft_data[NUM_TRAIN + NUM_VAL : TOTAL_NEEDED]

# 封装一个保存文件的辅助函数
def save_jsonl(data_list, filename):
    filepath = os.path.join(get_unc_path(DATA_OUTPUT_DIR), filename)
    with open(filepath, 'w', encoding='utf-8') as f:
        for item in data_list:
            # 必须使用 json.dumps 且 ensure_ascii=False 保留原始符号，每条记录占一行
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f"已保存: {filepath} ({len(data_list)} 条)")

# 执行保存
save_jsonl(train_data, 'App5_Data_Train.jsonl')
save_jsonl(val_data, 'App5_Data_Val.jsonl')
save_jsonl(test_data, 'App5_Data_Test.jsonl')

print("\n🎉 全部 SFT 数据集生成与拆分完成！App 5 数据预处理完成。")

目标生成 SFT 数据总量: 1210 条...


Production Generation:  67%|██████▋   | 1210/1815 [1:36:30<48:15,  4.79s/it]  


API 调用结束。成功收集 1210 条数据。正在进行切分...
已保存: \\?\e:\05_YZH_DS\02_Monash_DS\2024_S2_FIT5196_Data wrangling\5. assessments\FIT5196 assignment 1\04_github_submission\2024-07-Google-Map-Review-Analysis-and-Machine-Learning\01_data\05_data_for_app5\App5_Data_Train.jsonl (1000 条)
已保存: \\?\e:\05_YZH_DS\02_Monash_DS\2024_S2_FIT5196_Data wrangling\5. assessments\FIT5196 assignment 1\04_github_submission\2024-07-Google-Map-Review-Analysis-and-Machine-Learning\01_data\05_data_for_app5\App5_Data_Val.jsonl (200 条)
已保存: \\?\e:\05_YZH_DS\02_Monash_DS\2024_S2_FIT5196_Data wrangling\5. assessments\FIT5196 assignment 1\04_github_submission\2024-07-Google-Map-Review-Analysis-and-Machine-Learning\01_data\05_data_for_app5\App5_Data_Test.jsonl (10 条)

🎉 全部 SFT 数据集生成与拆分完成！App 5 数据预处理完成。


In [44]:
# 获取三个数据集的完整路径
train_path = os.path.join(get_unc_path(DATA_OUTPUT_DIR), 'App5_Data_Train.jsonl')
val_path = os.path.join(get_unc_path(DATA_OUTPUT_DIR), 'App5_Data_Val.jsonl')
test_path = os.path.join(get_unc_path(DATA_OUTPUT_DIR), 'App5_Data_Test.jsonl')

# 执行检查并打印第一条数据
inspect_jsonl_file(train_path, num_lines_to_print=1)
inspect_jsonl_file(val_path, num_lines_to_print=0) # 仅查看维度，不打印数据
inspect_jsonl_file(test_path, num_lines_to_print=0)

File: App5_Data_Train.jsonl
Total rows (Dimension): 1000
--------------------------------------------------
--- Sample 1 ---
{
    "gmap_id": "0x80c2c3713f0e96b9:0x72f1262a3b11cd02",
    "store_name": "Tiffany & Co.",
    "category": "Jewelry store, Gift shop, Jewelry designer, Jewelry engraver, Watch store",
    "description": "Luxury American retailer known for fine jewelry, china & silver, plus wedding registry.",
    "source_reviews_count": 10,
    "address": "Tiffany & Co., 68 W Colorado Blvd, Pasadena, CA 91105",
    "avg_rating": "4.3",
    "url": "https://www.google.com/maps/place//data=!4m2!3m1!1s0x80c2c3713f0e96b9:0x72f1262a3b11cd02?authuser=-1&hl=en&gl=us",
    "messages": [
        {
            "role": "system",
            "content": "You are a professional local business analyst. Please objectively and accurately answer user questions about a specific business based on the provided context. Be honest if you don't know."
        },
        {
            "role": "user",
  

## Step 7: 训练集文本长度分布分析 (Length Distribution Analysis)
**目标:** 通过统计训练集中单词和估算 Token 的数量分布，为微调阶段的 `max_length` 超参数提供数据支撑。
* **统计方法:** 拼接 `system`、`user` 和 `assistant` 的全部内容，计算单词总数。
* **Token 估算:** 按 `1 单词 ≈ 1.5 Tokens` 的保守比例估算最大上下文长度。
* **决策依据:** 观察 95% 和 99% 分位数，以确定在不截断核心信息的前提下，最节省显存的 `max_length` 设定值。

In [45]:
# ==========================================
# [Cell 7] 训练集文本长度分布分析
# ==========================================
import json
import pandas as pd
import os

train_file_path = os.path.join(get_unc_path(DATA_OUTPUT_DIR), 'App5_Data_Train.jsonl')

if not os.path.exists(train_file_path):
    print(f"[Error] 找不到文件: {train_file_path}")
else:
    length_stats = []
    
    with open(train_file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            
            # 将 system, user, assistant 的内容拼接在一起，模拟输入给大模型的完整上下文
            full_text = " ".join([msg['content'] for msg in data['messages']])
            
            # 计算单词数
            word_count = len(full_text.split())
            
            # 采用 1.5 倍的安全系数估算 Token 数量
            estimated_tokens = int(word_count * 1.5)
            
            length_stats.append({
                'word_count': word_count,
                'estimated_tokens': estimated_tokens
            })
            
    df_stats = pd.DataFrame(length_stats)
    
    # 打印分布统计量，重点观察 90%, 95%, 99% 分位数
    print("=== 训练集文本长度分布统计 (Train Set Length Distribution) ===")
    print(df_stats.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).round(2))

=== 训练集文本长度分布统计 (Train Set Length Distribution) ===
       word_count  estimated_tokens
count     1000.00           1000.00
mean       932.69           1398.78
std        420.41            630.63
min        313.00            469.00
50%        839.50           1259.00
75%       1112.75           1669.00
90%       1440.20           2160.30
95%       1726.40           2589.60
99%       2291.03           3436.05
max       3521.00           5281.00


**分析结论 (Analysis Conclusion):**  
根据长度分布统计，训练集数据的估算 Token 数量分布如下：
* **中位数 (50%)**: 1259 Tokens
* **95% 分位数**: 2589 Tokens
* **99% 分位数**: 3436 Tokens
* **最大值 (Max)**: 5281 Tokens

**超参数设定决策:**  
统计表明，超过 99% 的样本长度在 3436 Tokens 以内。因此，在后续微调阶段将 `max_length` 设定为 **4096** 为最优策略。该设定能够完整包含绝大多数样本的上下文信息。对于极少部分（不足 1%）超过 4096 Tokens 的极端离群数据，模型框架将自动执行末尾截断。这种处理方式在保证数据完整性的同时，有效规避了设置过大 `max_length` 导致的显存浪费与 OOM (Out Of Memory) 风险。

---
# END